# 02 — Shard anatomy: is a shard actually coherent?

Blog: [The Road to Cybernaut-1](https://nosible.com/blog/the-road-to-cybernaut-1) —
sections **A1 (the shard as the unit of everything)** and **A2 (shard artifacts)**.

The blog's central claim is that a shard is *not* a hash bucket. It is a semantically
and lexically coherent subset of the corpus, and that coherence is what makes the rest
of the design work:

- synonym expansion is unambiguous inside a coherent shard — "gene" cannot mean Gene
  Wilder in the CRISPR shard;
- a shard *summary* becomes a meaningful routing target for a dense vector.

Coherence is a claim, and claims can be measured. The build guide states the
acceptance criterion outright:

> Intra-shard mean cosine > inter-shard mean cosine by a clear margin.
> Every shard's document count lies within a configured band of the mean.

This notebook computes both, straight off the index the catalog hands us.

```mermaid
flowchart LR
    subgraph SHARD["one shard = a self-contained mini search engine"]
        M["Metadata<br/>title · summary"]
        K["Keywords<br/>TF-IDF vs corpus"]
        E["Entities<br/>NER counts"]
        C["Centroid<br/>L2-normalised mean"]
        B["BM25 index<br/>lexical"]
        V["Vectors<br/>semantic"]
        G["Term graph<br/>co-occurrence"]
    end
    Q(["query"]) --> C
    C --> SEL["shard selection"]
    K --> SEL
    E --> SEL
    G --> SEL
    SEL --> B
    SEL --> V
```

## Load the index through the catalog

`shard_index` is a single catalog entry covering the whole index directory. Its
`load` returns a query-ready `LoadedIndex` — documents, vectors, manifests and a
per-shard BM25 index — so the notebook never touches a path.

If the sample index has not been built yet, this builds it first (hash embedder, no
downloads, well under a second). An existing index is never overwritten.

In [ ]:
import numpy as np

from cybernaut_mini.notebook import ensure_fixture_index, kedro_catalog

ensure_fixture_index()
catalog = kedro_catalog(index_path="artifacts/fixture")

index = catalog.load("shard_index")

print(f"embedding model : {index.meta.embedding_model}")
print(f"embedding dim   : {index.meta.embedding_dim}")
print(f"documents       : {index.meta.n_documents}")
print(f"shards          : {index.meta.n_shards}")
print(f"seed            : {index.meta.seed}")
print(f"build revision  : {index.meta.embedding_revision or '(hash embedder — pure code)'}")

## Uniformity — are the shards evenly sized?

Uniformity is a *serving* property, separate from coherence: roughly equal shard
sizes make queries-per-second-per-shard predictable. The classic k-means failure
mode is one giant shard plus a long tail of singletons.

In [ ]:
sizes = np.array(
    [len(index.manifests[s].document_ids) for s in sorted(index.manifests)], dtype=float
)
mean = sizes.mean()

print(f"{'shard':>6} {'docs':>6} {'vs mean':>9}")
print("-" * 52)
for shard_id, size in enumerate(sizes):
    deviation = (size - mean) / mean * 100
    bar = "#" * int(size)
    print(f"{shard_id:>6} {int(size):>6} {deviation:>+8.1f}%   {bar}")

cv = sizes.std() / mean
worst = np.abs(sizes - mean).max() / mean

print()
print(f"mean shard size          : {mean:.1f}")
print(f"min / max                : {int(sizes.min())} / {int(sizes.max())}")
print(f"coefficient of variation : {cv:.3f}   (0 = perfectly uniform)")
print(f"worst deviation from mean: {worst * 100:.1f}%")
print()
verdict = "UNIFORM (within +/-50% band)" if worst <= 0.5 else "SKEWED (outside +/-50%)"
print("verdict:", verdict)

## Coherence — the actual measurement

Vectors in the index are L2-normalised, so cosine similarity is just a dot product
and the whole similarity matrix is one matmul.

- **intra-shard**: mean cosine between distinct documents *in the same* shard
- **inter-shard**: mean cosine between documents in *different* shards

The gap between them is the number that decides whether "coherent shard" is a real
property or marketing.

In [ ]:
def coherence(vectors: np.ndarray, labels: list[int]) -> tuple[float, float]:
    """Return (intra-shard mean cosine, inter-shard mean cosine).

    Self-similarity is excluded — every document matches itself at 1.0, which would
    inflate the intra figure by exactly the amount that makes it meaningless.
    """
    similarity = vectors @ vectors.T
    label_array = np.asarray(labels)
    same_shard = label_array[:, None] == label_array[None, :]
    off_diagonal = ~np.eye(len(labels), dtype=bool)

    intra_mask = same_shard & off_diagonal
    inter_mask = (~same_shard) & off_diagonal

    intra = float(similarity[intra_mask].mean()) if intra_mask.any() else float("nan")
    inter = float(similarity[inter_mask].mean()) if inter_mask.any() else float("nan")
    return intra, inter


# Recover each document's shard from the manifests, in index row order.
shard_of = {
    doc_id: manifest.shard_id
    for manifest in index.manifests.values()
    for doc_id in manifest.document_ids
}
labels = [shard_of[doc.id] for doc in index.documents]

intra, inter = coherence(index.vectors, labels)
margin = intra - inter

print(f"intra-shard mean cosine : {intra:.4f}")
print(f"inter-shard mean cosine : {inter:.4f}")
print(f"margin                  : {margin:+.4f}  ({margin / abs(inter) * 100:+.1f}% vs inter)")
print()
print("verdict:", "COHERENT — intra exceeds inter" if margin > 0 else "NOT COHERENT")

## Dynamic 1 — coherence and uniformity as `n_shards` varies

This is the knob with the most consequence in the whole build, and it is a genuine
trade-off rather than a value to maximise:

- **too few shards** — each one is internally diverse (low coherence), so shard
  selection cannot discriminate and every query fans out widely;
- **too many shards** — each is tiny and coherent, but routing has more candidates to
  rank and per-shard BM25 statistics get thin.

We re-cluster the *same* vectors at several shard counts. Nothing is rebuilt or
written; this reads the index once and clusters in memory.

In [ ]:
from cybernaut_mini.sharding import shard_documents

n_docs = len(index.documents)

header = f"{'n_shards':>9} {'intra':>8} {'inter':>8} {'margin':>9} {'size CV':>9}"
print(header + f" {'min':>5} {'max':>5}")
print("-" * 60)

sweep = []
for k in (2, 4, 8, 12, 16, 24, 32):
    if k > n_docs:
        continue
    result = shard_documents(index.vectors, n_shards=k, seed=42)
    k_intra, k_inter = coherence(index.vectors, result.labels)
    counts = np.bincount(result.labels, minlength=k).astype(float)
    k_cv = counts.std() / counts.mean()
    sweep.append((k, k_intra, k_inter, k_intra - k_inter, k_cv))
    print(
        f"{k:>9} {k_intra:>8.4f} {k_inter:>8.4f} {k_intra - k_inter:>+9.4f} "
        f"{k_cv:>9.3f} {int(counts.min()):>5} {int(counts.max()):>5}"
    )

best = max(sweep, key=lambda row: row[3])
print()
print(f"widest coherence margin at n_shards={best[0]} ({best[3]:+.4f})")
print("Note how the margin grows with k while shard sizes get less even — that is the trade-off.")

## Dynamic 2 — the margin only means something against a baseline

A positive margin is easy to get. The honest test is whether *this* clustering beats
a random assignment of the same documents into the same number of equally-sized
groups. Random shards should show a margin near zero, because two documents landing
together tells you nothing about their content.

In [ ]:
rng = np.random.default_rng(42)
n_shards = index.meta.n_shards

random_margins = []
for _trial in range(20):
    random_labels = list(rng.permutation(np.arange(n_docs) % n_shards))
    r_intra, r_inter = coherence(index.vectors, [int(x) for x in random_labels])
    random_margins.append(r_intra - r_inter)

random_margins = np.array(random_margins)

print(f"learned shards  margin : {margin:+.4f}")
print(f"random shards   margin : {random_margins.mean():+.4f} "
      f"(sd {random_margins.std():.4f}, n=20 trials)")
print()
if random_margins.std() > 0:
    z = (margin - random_margins.mean()) / random_margins.std()
    print(f"the learned margin is {z:.1f} standard deviations above random")
print()
print("This is the control that makes the coherence claim falsifiable rather than decorative.")

## Inside one shard

Every artifact below is *rebuildable* from `(corpus, config, seed)`. Nothing here is
hand-edited, and nothing needs to be: the index directory is reproducible byte-for-byte.

In [ ]:
# The most coherent shard: highest mean cosine to its own centroid.
def shard_tightness(shard_id: int) -> float:
    manifest = index.manifests[shard_id]
    rows = [index.row_map[d] for d in manifest.document_ids]
    centroid = np.asarray(manifest.centroid, dtype=np.float32)
    return float((index.vectors[rows] @ centroid).mean())


tightest = max(index.manifests, key=shard_tightness)
manifest = index.manifests[tightest]

print(f"shard {manifest.shard_id:03d}  ({manifest.document_count} documents, "
      f"tightness {shard_tightness(tightest):.4f})")
print(f"title    : {manifest.title}")
print(f"summary  : {manifest.summary}")
print()
print("top keywords (TF-IDF of the term in this shard vs the corpus):")
for keyword in manifest.keywords[:10]:
    print(f"    {keyword.weight:>8.4f}  {keyword.term}")
print()
print("entities:", ", ".join(e.text for e in manifest.entities[:10]) or "(none — regex backend)")
print()
print("member documents:")
for doc_id in manifest.document_ids[:8]:
    print(f"    {doc_id}  {index.by_id[doc_id].title[:64]}")

### Do the keywords actually separate the shards?

If shards were incoherent, their top keywords would look alike. Overlap between
top-10 keyword sets is a cheap, direct check on that.

In [ ]:
top_terms = {
    shard_id: {kw.term for kw in m.keywords[:10]} for shard_id, m in index.manifests.items()
}

shard_ids = sorted(top_terms)
print("pairwise top-10 keyword overlap (Jaccard)")
print("      " + "".join(f"{s:>7}" for s in shard_ids))
overlaps = []
for a in shard_ids:
    row = f"{a:>4}  "
    for b in shard_ids:
        if a == b:
            row += f"{'—':>7}"
            continue
        union = top_terms[a] | top_terms[b]
        jaccard = len(top_terms[a] & top_terms[b]) / len(union) if union else 0.0
        overlaps.append(jaccard)
        row += f"{jaccard:>7.2f}"
    print(row)

print()
print(f"mean pairwise overlap: {np.mean(overlaps):.3f}   (0 = every shard has its own vocabulary)")

## What is *not* built here

The build guide is explicit that this replica is a subset. Being honest about the gap
is part of the teaching value:

| Blog artifact | Status in `cybernaut-mini` |
|---|---|
| Metadata, keywords, entities, centroid, BM25, vectors, term graph | Built — everything measured above |
| Compression dictionary | Partial — generic zlib, no trained zstd dictionary |
| Bloom filters | **Not built** — approximated by keyword + graph coverage |
| LLM-generated example queries | **Not built** |
| Per-shard evals | Partial — global judgments only |
| Write-ahead log | **Not built** |

Next: [`03_retrieval_evaluation.ipynb`](03_retrieval_evaluation.ipynb) puts the index
to work and measures whether hybrid retrieval actually beats its parts.